# Ejercicio 1: Replicación del ADN y Extensión con Biopython

## 📖 Introducción y Fundamento Teórico Resumido

Antes de programar la solución, repasemos los conceptos fundamentales del ejercicio:

1. **Secuencia de partida:** Se nos proporciona el dúplex de ADN bicatenario:
   $$\begin{aligned}
   \text{Hebra superior (sentido):} \quad & 5'\text{ – ATG CCG TTA GCT – }3' \\
   \text{Hebra inferior (molde):} \quad & 3'\text{ – TAC GGC AAT CGA – }5'
   \end{aligned}$$
   Ambas hebras son **antiparalelas** ($5' \to 3'$ frente a $3' \to 5'$) y sus bases están unidas por puentes de hidrógeno según las reglas de Watson y Crick ($A=T$ con 2 enlaces, $G \equiv C$ con 3 enlaces).

2. **Mecanismo semiconservativo (Meselson y Stahl, 1958):**
   Durante la replicación, la doble hélice se abre y cada hebra parental sirve de molde para sintetizar una nueva hebra complementaria. Al finalizar 1 ronda, surgen **2 moléculas hijas bicatenarias idénticas**, cada una con **una hebra parental intacta y una hebra de nueva síntesis**.

3. **Maquinaria del replisoma:**
   - **Helicasa:** Rompe los puentes de hidrógeno abriendo la doble hélice (gasto de ATP).
   - **Primasa:** Sintetiza un cebador corto de ARN que aporta el extremo $3'-OH$ libre.
   - **ADN Polimerasa:** Sintetiza la nueva cadena en dirección $5' \to 3'$ y corrige errores mediante su actividad exonucleasa $3' \to 5'$ (*proofreading*).
   - **Ligasa:** Sella las mellas (*nicks*) uniendo los fragmentos de Okazaki.

4. **Consecuencia de un error no corregido:**
   Un nucleótido mal apareado no reparado se fija permanentemente como **mutación puntual** tras la siguiente ronda replicativa, pudiendo ser silenciosa, de cambio de sentido (*missense*), o sin sentido (*nonsense*, parada prematura).

# 💻 Extensión con Biopython: Guía Paso a Paso

> **Enunciado:** *Escribe un pequeño script que, dada una cadena de ADN, genere automáticamente su hebra complementaria, y comprueba si tu resultado coincide con lo que obtuviste manualmente.*

A continuación, construiremos y ejecutaremos el script dividiéndolo en pequeños bloques interactivos comentados.

### Paso 1: Importar la clase `Seq` de Biopython

En Python estándar, una secuencia podría guardarse como un simple string (`str`). Sin embargo, Biopython ofrece la clase especializada **`Seq`** (dentro del módulo `Bio.Seq`), la cual incorpora métodos biológicos nativos (como `.complement()`, `.reverse_complement()`, `.transcribe()`, `.translate()`).

Ejecuta la siguiente celda para importar `Seq` y verificar que el entorno está listo:

In [2]:
import Bio
from Bio.Seq import Seq

print(f"Biopython versión: {Bio.__version__}")
print("Módulo Bio.Seq cargado correctamente.")

Biopython versión: 1.88
Módulo Bio.Seq cargado correctamente.


### Paso 2: Definir la secuencia parental de ADN dada (5' -> 3')

Tomamos la cadena parental de 12 nucleótidos: `ATGCCGTTAGCT`.

La convertimos en un objeto `Seq` y examinamos sus propiedades básicas (longitud, extremos y tipo de dato en Python).

In [3]:
# Definición de la secuencia original (5' -> 3')
seq_5_to_3_str = "ATGCCGTTAGCT"
dna_seq = Seq(seq_5_to_3_str)

print("=" * 60)
print("SECUENCIA PARENTAL ORIGINAL")
print("=" * 60)
print(f"Secuencia (texto):                 {dna_seq}")
print(f"Con polaridad:                     5'- {dna_seq} -3'")
print(f"Longitud:                          {len(dna_seq)} nucleótidos (pb)")
print(f"Tipo de objeto en Python:          {type(dna_seq)}")
print("=" * 60)

SECUENCIA PARENTAL ORIGINAL
Secuencia (texto):                 ATGCCGTTAGCT
Con polaridad:                     5'- ATGCCGTTAGCT -3'
Longitud:                          12 nucleótidos (pb)
Tipo de objeto en Python:          <class 'Bio.Seq.Seq'>


### Paso 3: Generación de la hebra complementaria con `.complement()`

El método **`dna_seq.complement()`** de Biopython aplica las reglas de apareamiento de Watson y Crick base por base en la misma posición de índice ($A \leftrightarrow T$ y $C \leftrightarrow G$):

$$\begin{aligned}
\text{Hebra parental } (5' \to 3'): & \quad 5'\text{- A T G C C G T T A G C T -}3' \\
\text{Hebra complementaria } (3' \to 5'): & \quad 3'\text{- T A C G G C A A T C G A -}5'
\end{aligned}$$

Nótese que, al no alterar el orden de los índices, la hebra resultante generada por `.complement()` mantiene la **orientación física antiparalela $3' \to 5'$**.

In [4]:
# Generamos la hebra complementaria directa (orientación 3' -> 5')
dna_complement = dna_seq.complement()

print("=" * 60)
print("HEBRA COMPLEMENTARIA GENERADA CON .complement()")
print("=" * 60)
print(f"Hebra parental       (5' -> 3'): 5'- {dna_seq} -3'")
print(f"Hebra complementaria (3' -> 5'): 3'- {dna_complement} -5'")
print("=" * 60)

HEBRA COMPLEMENTARIA GENERADA CON .complement()
Hebra parental       (5' -> 3'): 5'- ATGCCGTTAGCT -3'
Hebra complementaria (3' -> 5'): 3'- TACGGCAATCGA -5'


### Paso 4: La hebra complementaria en la convención estándar (5' -> 3') con `.reverse_complement()`

En las publicaciones científicas, las bases de datos moleculares (como GenBank o Ensembl) y la síntesis de cebadores, **toda secuencia se reporta obligatoriamente en sentido $5' \to 3'$**.

Para leer la hebra complementaria en dirección $5' \to 3'$, debemos leerla de derecha a izquierda (invertirla):

$$\text{Hebra complementaria } (3' \to 5'): \quad \text{T A C G G C A A T C G A}$$
$$\Downarrow \quad (\text{lectura invertida } 5' \to 3')$$
$$\text{Hebra reversa complementaria } (5' \to 3'): \quad \text{A G C T A A C G G C A T}$$

Biopython automatiza este proceso mediante **`dna_seq.reverse_complement()`**.

In [5]:
# Generamos la reversa complementaria (sentido biológico 5' -> 3')
dna_rev_complement = dna_seq.reverse_complement()

print("=" * 60)
print("REVERSA COMPLEMENTARIA (CONVENCIÓN ESTÁNDAR 5' -> 3')")
print("=" * 60)
print(f"Orientación estándar (5' -> 3'): {dna_rev_complement}")
print(f"Con extremos marcados:           5'- {dna_rev_complement} -3'")
print("=" * 60)

REVERSA COMPLEMENTARIA (CONVENCIÓN ESTÁNDAR 5' -> 3')
Orientación estándar (5' -> 3'): AGCTAACGGCAT
Con extremos marcados:           5'- AGCTAACGGCAT -3'


### Paso 5: Comprobación automática frente al resultado manual

El enunciado nos pide: *«comprueba si tu resultado coincide con lo que obtuviste manualmente»*.

En el análisis teórico previo dedujimos manualmente:
- Complementaria ($3' \to 5'$): `TACGGCAATCGA`
- Reversa complementaria ($5' \to 3'$): `AGCTAACGGCAT`

En Python, la forma más rigurosa de validar que un resultado algorítmico coincide de forma exacta con la expectativa teórica es utilizando **aserciones (`assert`)**. Si alguna letra no coincide, el intérprete lanzará un `AssertionError` advirtiendo la discrepancia.

In [6]:
# Valores obtenidos manualmente en la resolución teórica previa:
manual_complement_3_5 = "TACGGCAATCGA"
manual_rev_complement_5_3 = "AGCTAACGGCAT"

# Verificación estricta mediante aserciones:
assert str(dna_complement) == manual_complement_3_5, (
    f"Error en complementaria: esperado {manual_complement_3_5}, obtenido {dna_complement}"
)
assert str(dna_rev_complement) == manual_rev_complement_5_3, (
    f"Error en reversa complementaria: esperado {manual_rev_complement_5_3}, obtenido {dna_rev_complement}"
)

print("=" * 65)
print(">> VERIFICACIÓN MANUAL vs BIOPYTHON: EXITOSA (COINCIDENCIA EXACTA)")
print(f"   [OK] Complementaria (3'->5'):   {dna_complement} == {manual_complement_3_5}")
print(f"   [OK] Reversa compl. (5'->3'):   {dna_rev_complement} == {manual_rev_complement_5_3}")
print("=" * 65)

>> VERIFICACIÓN MANUAL vs BIOPYTHON: EXITOSA (COINCIDENCIA EXACTA)
   [OK] Complementaria (3'->5'):   TACGGCAATCGA == TACGGCAATCGA
   [OK] Reversa compl. (5'->3'):   AGCTAACGGCAT == AGCTAACGGCAT


### Paso 6: Esquematización de la replicación semiconservativa con Python

Aprovechando los objetos `dna_seq` y `dna_complement`, podemos programar un bloque que visualice dinámicamente cómo las dos hebras del dúplex parental se separan y originan las dos moléculas hijas dúplex tras una ronda de replicación.

In [7]:
# Esquema generado con Python a partir de las secuencias del script
print("=" * 65)
print("ESQUEMA DINÁMICO DE REPLICACIÓN SEMICONSERVATIVA (1 RONDA)")
print("=" * 65)
print("\n1. Dúplex parental original:")
print(f"   5'- {dna_seq} -3'")
print(f"   3'- {dna_complement} -5'")

print("\n        |")
print("        v  (Separación por Helicasa y síntesis por ADN Polimerasa)")

print("\n2. Moléculas hijas resultantes tras la ronda de replicación:")
print("  +-------------------------------------------------------------+")
print("  | Molécula Hija 1:                                            |")
print(f"  |   [Parental]     5'- {dna_seq} -3'                      |")
print(f"  |   [Nueva hebra]  3'- {dna_complement} -5'                      |")
print("  +-------------------------------------------------------------+")
print("  | Molécula Hija 2:                                            |")
print(f"  |   [Nueva hebra]  5'- {dna_seq} -3'                      |")
print(f"  |   [Parental]     3'- {dna_complement} -5'                      |")
print("  +-------------------------------------------------------------+")
print("=" * 65)

ESQUEMA DINÁMICO DE REPLICACIÓN SEMICONSERVATIVA (1 RONDA)

1. Dúplex parental original:
   5'- ATGCCGTTAGCT -3'
   3'- TACGGCAATCGA -5'

        |
        v  (Separación por Helicasa y síntesis por ADN Polimerasa)

2. Moléculas hijas resultantes tras la ronda de replicación:
  +-------------------------------------------------------------+
  | Molécula Hija 1:                                            |
  |   [Parental]     5'- ATGCCGTTAGCT -3'                      |
  |   [Nueva hebra]  3'- TACGGCAATCGA -5'                      |
  +-------------------------------------------------------------+
  | Molécula Hija 2:                                            |
  |   [Nueva hebra]  5'- ATGCCGTTAGCT -3'                      |
  |   [Parental]     3'- TACGGCAATCGA -5'                      |
  +-------------------------------------------------------------+


### Paso 7 (Ampliación interactiva): Función modular para cualquier secuencia de ADN

Para hacer el script reutilizable y extensible a cualquier problema bioinformático futuro, encapsulamos la lógica en una función `analizar_replicacion_adn(secuencia)`.

Esta función:
1. Valida que la secuencia solo contenga nucleótidos válidos de ADN (`A, C, G, T`).
2. Calcula automáticamente ambas hebras complementarias.
3. Muestra el reporte completo y el esquema de replicación.

In [8]:
def analizar_replicacion_adn(secuencia_dna_str: str) -> dict:
    """
    Dada una cadena de ADN (5' -> 3'), calcula sus hebras complementarias
    e imprime el reporte completo de replicación semiconservativa.
    """
    secuencia_limpia = secuencia_dna_str.strip().upper()
    
    # Validación básica de nucleótidos de ADN
    bases_validas = set("ACGT")
    if not set(secuencia_limpia).issubset(bases_validas):
        caracteres_invalidos = set(secuencia_limpia) - bases_validas
        raise ValueError(f"Secuencia contiene caracteres no válidos para ADN: {caracteres_invalidos}")
    
    seq = Seq(secuencia_limpia)
    comp_3_5 = seq.complement()
    rev_comp_5_3 = seq.reverse_complement()
    
    print(f"\n>>> Análisis de Replicación para: {seq}")
    print(f"    Longitud:                {len(seq)} pb")
    print(f"    Molde parental (5'->3'): 5'- {seq} -3'")
    print(f"    Complementaria (3'->5'): 3'- {comp_3_5} -5'")
    print(f"    Reversa compl. (5'->3'): 5'- {rev_comp_5_3} -3'")
    
    return {
        "parental": seq,
        "complementaria_3_5": comp_3_5,
        "reversa_complementaria_5_3": rev_comp_5_3
    }

# Probamos la función con la secuencia del Ejercicio 1:
resultado = analizar_replicacion_adn("ATGCCGTTAGCT")

# Y podemos probarla con cualquier otra secuencia que desees:
resultado_extra = analizar_replicacion_adn("GATTACA")


>>> Análisis de Replicación para: ATGCCGTTAGCT
    Longitud:                12 pb
    Molde parental (5'->3'): 5'- ATGCCGTTAGCT -3'
    Complementaria (3'->5'): 3'- TACGGCAATCGA -5'
    Reversa compl. (5'->3'): 5'- AGCTAACGGCAT -3'

>>> Análisis de Replicación para: GATTACA
    Longitud:                7 pb
    Molde parental (5'->3'): 5'- GATTACA -3'
    Complementaria (3'->5'): 3'- CTAATGT -5'
    Reversa compl. (5'->3'): 5'- TGTAATC -3'


## 📌 Resumen de lo aprendido

- **`Seq.complement()`:** Cambia bases según Watson-Crick conservando los índices ($3' \to 5'$).
- **`Seq.reverse_complement()`:** Cambia bases e invierte el orden para presentar la cadena en la convención estándar $5' \to 3'$.
- **`assert`:** Permite validar programáticamente que los cálculos computacionales replican la resolución teórica manual sin errores.
- Las moléculas dúplex hijas de la replicación son siempre idénticas a la molécula original, conservando fielmente la información genética.